# Phase 2: White-Box Membership Inference

Develop on the labeled `train` target, then generate membership scores for the hidden-label `val` target. A larger score must mean that an example is more likely to be a member.

## Configuration

Keep `PHASE = "train"` while developing. Change it to `"val"` only when generating the Phase 2 submission. The `final` target is reserved for the later project phase.

In [ ]:
import json
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from peft import PeftModel
from sklearn.metrics import roc_auc_score, roc_curve
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
PHASE = "train"  # train, val, or final
BATCH_SIZE = 8
MAX_LENGTH = 512
MODEL_NAME = "gpt2_3_lora32_adamw_b8_lr2"

if PHASE not in {"train", "val", "final"}:
    raise ValueError(f"Unknown phase: {PHASE}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / PHASE
ADAPTER_DIR = ROOT / "models" / PHASE / MODEL_NAME


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    model_dtype = torch.float16
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    model_dtype = torch.float32
else:
    device = torch.device("cpu")
    model_dtype = torch.float32

print(f"Phase: {PHASE}")
print(f"Device: {device}")
print(f"Adapter: {ADAPTER_DIR}")


## Load the target

Each target is a separate LoRA adapter over the public GPT-2 base model. The first run may download the base model from Hugging Face.

In [ ]:
def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

test_items = read_json(DATA_DIR / "test.json")
if len(test_items) != 2000 or not all(isinstance(text, str) for text in test_items):
    raise ValueError("Expected 2,000 test strings")

adapter_config = read_json(ADAPTER_DIR / "adapter_config.json")
base_model_name = adapter_config["base_model_name_or_path"]

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=model_dtype,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)
model.eval()

print(f"Loaded {len(test_items)} test examples")


## Implement the attack

Return one finite score for every input text. You may reuse Phase 1 features, train shadow models, calibrate scores, or combine multiple signals. Do not read labels inside the attack.

In [ ]:
@torch.inference_mode()
def your_attack(model, tokenizer, texts, device, batch_size, max_length):
    """Return one membership score per text; larger means more likely member."""
    # Implement your Phase 2 attack here.
    raise NotImplementedError("Implement your_attack before continuing")


In [ ]:
scores_test = np.asarray(
    your_attack(
        model=model,
        tokenizer=tokenizer,
        texts=test_items,
        device=device,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
    ),
    dtype=np.float64,
)

if scores_test.shape != (len(test_items),):
    raise ValueError(f"Expected {len(test_items)} scores, got shape {scores_test.shape}")
if not np.isfinite(scores_test).all():
    raise ValueError("Attack scores contain NaN or infinity")

print(f"Generated {len(scores_test)} scores")
print(f"Score range: [{scores_test.min():.6g}, {scores_test.max():.6g}]")


## Evaluate on the public target

This cell reads labels only when `PHASE == "train"`. The competition metric uses the strict condition `FPR < 0.01`.

In [ ]:
if PHASE == "train":
    labels = np.asarray(read_json(DATA_DIR / "test_label.json"), dtype=np.int64)
    fpr, tpr, thresholds = roc_curve(labels, scores_test)
    auc_value = roc_auc_score(labels, scores_test)
    tpr_at_one_percent = float(np.max(tpr[fpr < 0.01]))

    print(f"ROC AUC: {auc_value:.6f}")
    print(f"TPR at FPR < 0.01: {tpr_at_one_percent:.6f}")

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"Attack (AUC={auc_value:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Membership Inference ROC")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("Hidden-label phase: local evaluation is unavailable.")


## Export and package predictions

For Phase 2, set `PHASE = "val"`, rerun the notebook, and execute this cell. It creates a ZIP with `val/prediction.csv` at the archive root and runs the same structural checks described in the README.

In [ ]:
if PHASE == "train":
    print("No submission is created for the labeled train target.")
else:
    prediction_path = DATA_DIR / "prediction.csv"
    with prediction_path.open("w", encoding="utf-8") as handle:
        for score in scores_test:
            handle.write(f"{float(score):.17g}\n")

    output_path = ROOT / f"project_phase2_{PHASE}_submission.zip"
    subprocess.run(
        [
            sys.executable,
            str(ROOT / "scripts" / "make_submission.py"),
            "--phase",
            PHASE,
            "--prediction",
            str(prediction_path),
            "--output",
            str(output_path),
        ],
        check=True,
    )
    print(f"Ready to upload: {output_path}")
